# Объединение базовой модели и LoRA-адаптера

## Импорты

In [1]:
from pathlib import Path
import gc
import json
import platform
import sys

import pandas as pd
import torch
import transformers
import peft
from peft import PeftModel
from transformers import AutoConfig, AutoProcessor, Qwen2_5_VLForConditionalGeneration

c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Настройки

In [2]:
def find_root():
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "models").exists() and (folder / "outputs").exists():
            return folder
    raise FileNotFoundError("Корень проекта не найден")


ROOT = find_root()

MODEL_VARIANTS = {
    "qwen_lora_1": {
        "family": "qwen",
        "base_model": ROOT / "models" / "Qwen2.5-VL-3B-Instruct",
        "adapter": ROOT / "outputs" / "qwen_lora_adapter",
        "output": ROOT / "models" / "Qwen2.5-VL-3B-Instruct-LoRA-1-merged",
    },
    "qwen_lora_2": {
        "family": "qwen",
        "base_model": ROOT / "models" / "Qwen2.5-VL-3B-Instruct",
        "adapter": ROOT / "outputs" / "qwen_lora_adapter_2",
        "output": ROOT / "models" / "Qwen2.5-VL-3B-Instruct-LoRA-2-merged",
    },
    "qwen_lora_3": {
        "family": "qwen",
        "base_model": ROOT / "models" / "Qwen2.5-VL-3B-Instruct",
        "adapter": ROOT / "outputs" / "qwen_lora_adapter_3",
        "output": ROOT / "models" / "Qwen2.5-VL-3B-Instruct-LoRA-3-merged",
    },
    "smolvlm_lora": {
        "family": "smolvlm",
        "base_model": ROOT / "models" / "SmolVLM-Instruct",
        "adapter": ROOT / "outputs" / "smolvlm_lora_adapter",
        "output": ROOT / "models" / "SmolVLM-Instruct-LoRA-merged",
    },
}

SELECTED_MODEL = "qwen_lora_2"
MAX_SHARD_SIZE = "4GB"
SAFE_SERIALIZATION = True

config = MODEL_VARIANTS[SELECTED_MODEL]
BASE_MODEL_DIR = config["base_model"]
ADAPTER_DIR = config["adapter"]
OUTPUT_DIR = config["output"]

## Проверка путей

In [3]:
if not BASE_MODEL_DIR.exists():
    raise FileNotFoundError(f"Базовая модель не найдена: {BASE_MODEL_DIR}")

if not ADAPTER_DIR.exists():
    raise FileNotFoundError(f"LoRA-адаптер не найден: {ADAPTER_DIR}")

if not (ADAPTER_DIR / "adapter_config.json").exists():
    raise FileNotFoundError(f"В папке нет adapter_config.json: {ADAPTER_DIR}")

if not (ADAPTER_DIR / "adapter_model.safetensors").exists():
    raise FileNotFoundError(f"В папке нет adapter_model.safetensors: {ADAPTER_DIR}")

if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise FileExistsError(f"Папка результата уже существует и не пустая: {OUTPUT_DIR}")

with open(ADAPTER_DIR / "adapter_config.json", encoding="utf-8") as file:
    adapter_config = json.load(file)

paths = pd.DataFrame([
    {"Параметр": "Вариант", "Значение": SELECTED_MODEL},
    {"Параметр": "Базовая модель", "Значение": str(BASE_MODEL_DIR)},
    {"Параметр": "LoRA-адаптер", "Значение": str(ADAPTER_DIR)},
    {"Параметр": "Готовая модель", "Значение": str(OUTPUT_DIR)},
    {"Параметр": "LoRA rank", "Значение": adapter_config.get("r")},
    {"Параметр": "LoRA alpha", "Значение": adapter_config.get("lora_alpha")},
])
paths

,Параметр,Значение
0,Вариант,qwen_lora_2
1,Базовая модель,c:\Users\user\Desktop\Практика\vk-vision-langu...
2,LoRA-адаптер,c:\Users\user\Desktop\Практика\vk-vision-langu...
3,Готовая модель,c:\Users\user\Desktop\Практика\vk-vision-langu...
4,LoRA rank,16
5,LoRA alpha,32


## Окружение

In [4]:
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    MODEL_DTYPE = torch.bfloat16
elif torch.cuda.is_available():
    MODEL_DTYPE = torch.float16
else:
    MODEL_DTYPE = torch.float32

DEVICE_MAP = "auto" if torch.cuda.is_available() else {"": "cpu"}

environment = pd.DataFrame([
    {"Параметр": "Python", "Значение": sys.version.split()[0]},
    {"Параметр": "Windows", "Значение": platform.platform()},
    {"Параметр": "PyTorch", "Значение": torch.__version__},
    {"Параметр": "Transformers", "Значение": transformers.__version__},
    {"Параметр": "PEFT", "Значение": peft.__version__},
    {"Параметр": "CUDA", "Значение": torch.cuda.is_available()},
    {"Параметр": "dtype", "Значение": str(MODEL_DTYPE)},
])
environment

,Параметр,Значение
0,Python,3.11.0
1,Windows,Windows-10-10.0.26200-SP0
2,PyTorch,2.5.1+cu121
3,Transformers,5.14.0.dev0
4,PEFT,0.19.1
5,CUDA,True
6,dtype,torch.bfloat16


## Класс модели

In [5]:
if config["family"] == "qwen":
    ModelClass = Qwen2_5_VLForConditionalGeneration
else:
    try:
        from transformers import AutoModelForImageTextToText as ModelClass
    except ImportError:
        try:
            from transformers import AutoModelForVision2Seq as ModelClass
        except ImportError:
            from transformers import Idefics3ForConditionalGeneration as ModelClass

ModelClass.__name__

'Qwen2_5_VLForConditionalGeneration'

## Загрузка базовой модели

In [6]:
load_kwargs = {
    "local_files_only": True,
    "low_cpu_mem_usage": True,
    "device_map": DEVICE_MAP,
}

try:
    base_model = ModelClass.from_pretrained(
        str(BASE_MODEL_DIR),
        dtype=MODEL_DTYPE,
        **load_kwargs,
    )
except TypeError:
    base_model = ModelClass.from_pretrained(
        str(BASE_MODEL_DIR),
        torch_dtype=MODEL_DTYPE,
        **load_kwargs,
    )

base_model.eval()
print(type(base_model).__name__)

Loading weights: 100%|██████████| 824/824 [00:09<00:00, 88.89it/s] 


Qwen2_5_VLForConditionalGeneration


## Подключение LoRA

In [7]:
model_with_lora = PeftModel.from_pretrained(
    base_model,
    str(ADAPTER_DIR),
    is_trainable=False,
    local_files_only=True,
)
model_with_lora.eval()
print(type(model_with_lora).__name__)

PeftModelForCausalLM


## Объединение весов

In [8]:
merged_model = model_with_lora.merge_and_unload(safe_merge=True)
merged_model.eval()
print(type(merged_model).__name__)

Qwen2_5_VLForConditionalGeneration


## Сохранение модели

In [9]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

merged_model.save_pretrained(
    str(OUTPUT_DIR),
    safe_serialization=SAFE_SERIALIZATION,
    max_shard_size=MAX_SHARD_SIZE,
)

processor_source = ADAPTER_DIR if (ADAPTER_DIR / "processor_config.json").exists() else BASE_MODEL_DIR
processor = AutoProcessor.from_pretrained(str(processor_source), local_files_only=True)
processor.save_pretrained(str(OUTPUT_DIR))

print(OUTPUT_DIR)

Writing model shards: 100%|██████████| 2/2 [00:06<00:00,  3.13s/it]


c:\Users\user\Desktop\Практика\vk-vision-language-vqa\models\Qwen2.5-VL-3B-Instruct-LoRA-2-merged


## Очистка памяти

In [10]:
del model_with_lora
del base_model
del merged_model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Проверка результата

In [11]:
saved_config = AutoConfig.from_pretrained(str(OUTPUT_DIR), local_files_only=True)
model_files = sorted(OUTPUT_DIR.glob("*.safetensors"))
total_size_gb = sum(file.stat().st_size for file in model_files) / 1024**3

result = pd.DataFrame([
    {"Параметр": "Папка", "Значение": str(OUTPUT_DIR)},
    {"Параметр": "Архитектура", "Значение": saved_config.architectures[0]},
    {"Параметр": "Файлов весов", "Значение": len(model_files)},
    {"Параметр": "Размер весов, GB", "Значение": round(total_size_gb, 2)},
    {"Параметр": "config.json", "Значение": (OUTPUT_DIR / "config.json").exists()},
    {"Параметр": "processor_config.json", "Значение": (OUTPUT_DIR / "processor_config.json").exists()},
    {"Параметр": "adapter_config.json", "Значение": (OUTPUT_DIR / "adapter_config.json").exists()},
])

if not model_files:
    raise FileNotFoundError("Файлы весов не были сохранены")

if (OUTPUT_DIR / "adapter_config.json").exists():
    raise RuntimeError("В результате осталась конфигурация адаптера")

result

,Параметр,Значение
0,Папка,c:\Users\user\Desktop\Практика\vk-vision-langu...
1,Архитектура,Qwen2_5_VLForConditionalGeneration
2,Файлов весов,2
3,"Размер весов, GB",6.99
4,config.json,True
5,processor_config.json,True
6,adapter_config.json,False


## Загрузка готовой модели

In [12]:
processor = AutoProcessor.from_pretrained(str(OUTPUT_DIR), local_files_only=True)

try:
    model = ModelClass.from_pretrained(
        str(OUTPUT_DIR),
        dtype=MODEL_DTYPE,
        device_map=DEVICE_MAP,
        local_files_only=True,
        low_cpu_mem_usage=True,
    )
except TypeError:
    model = ModelClass.from_pretrained(
        str(OUTPUT_DIR),
        torch_dtype=MODEL_DTYPE,
        device_map=DEVICE_MAP,
        local_files_only=True,
        low_cpu_mem_usage=True,
    )

model.eval()
print(type(model).__name__)
print(type(processor).__name__)

Loading weights: 100%|██████████| 824/824 [00:02<00:00, 395.37it/s]


Qwen2_5_VLForConditionalGeneration
Qwen2_5_VLProcessor
